# ClauseGuard — Structured Legal Information Extraction with NuExtract 2.0 2B

This notebook demonstrates the extraction layer of the **ClauseGuard** pipeline using **NuExtract 2.0 2B** (`AutoModelForVision2Seq` / `Qwen2VLForConditionalGeneration` + `AutoProcessor`).

### Pipeline Flow:
```
Clause Text + Predicted Category
    ↓
Category → Family Mapping & Category-Specific Schema
    ↓
NuExtract 2.0 2B Model Inference
    ↓
JSON Extraction
    ↓
Validation (JSON repair, Schema checking, Type enforcement)
    ↓
Normalization (Duration parsing e.g. '12 months' → {value: 12, unit: 'month'})
    ↓
Structured Clause Representation (with placeholder relationships: [])
```

In [ ]:
import os
import sys
import re
import json
import warnings
import torch
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

# Suppress HuggingFace Hub symlink and deprecation warnings on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Check hardware device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] PyTorch version: {torch.__version__}")
print(f"[*] Active Device:    {device.upper()}")

In [ ]:
# Category -> Family Mapping
CATEGORY_TO_FAMILY: Dict[str, str] = {
    "confidentiality": "confidentiality_ip",
    "ip_assignment": "confidentiality_ip",
    "exclusivity": "restrictive_covenant",
    "no_solicit_customers": "restrictive_covenant",
    "no_solicit_employees": "restrictive_covenant",
    "non_compete": "restrictive_covenant",
    "governing_law": "legal_contractual",
    "liability_indemnity": "legal_contractual",
    "termination_notice": "legal_contractual",
    "other_general": "general"
}

# 10 Category Schemas
EXTRACTION_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "confidentiality": {
        "protected_information": [],
        "obligations": [],
        "duration": None,
        "post_termination": None,
        "exceptions": []
    },
    "exclusivity": {
        "exclusive_to": None,
        "restricted_activities": [],
        "duration": None,
        "exceptions": []
    },
    "governing_law": {
        "jurisdiction": None,
        "governing_law": None
    },
    "ip_assignment": {
        "assigned_rights": [],
        "assignor": None,
        "assignee": None,
        "scope": None,
        "exceptions": []
    },
    "liability_indemnity": {
        "indemnifying_party": None,
        "protected_party": None,
        "covered_claims": [],
        "covered_damages": [],
        "limitations": [],
        "exceptions": []
    },
    "no_solicit_customers": {
        "restricted_action": None,
        "restricted_entities": [],
        "duration": None,
        "geographical_scope": None,
        "trigger": None,
        "exceptions": []
    },
    "no_solicit_employees": {
        "restricted_action": None,
        "restricted_entities": [],
        "duration": None,
        "geographical_scope": None,
        "trigger": None,
        "exceptions": []
    },
    "non_compete": {
        "restricted_activity": None,
        "competitors": [],
        "duration": None,
        "geographical_scope": None,
        "trigger": None,
        "exceptions": []
    },
    "other_general": {
        "obligations": [],
        "rights": [],
        "conditions": [],
        "entities": []
    },
    "termination_notice": {
        "notice_period": None,
        "notice_given_by": None,
        "termination_type": None,
        "conditions": [],
        "effective_date": None
    }
}

print(f"[✓] Configured {len(EXTRACTION_SCHEMAS)} category schemas and {len(CATEGORY_TO_FAMILY)} category-to-family mappings.")

In [ ]:
class NuExtractRunner:
    """
    Wrapper for NuExtract 2.0 2B model loading & inference.
    Supports Qwen2-VL architecture via Qwen2VLForConditionalGeneration / AutoModelForConditionalGeneration.
    """
    def __init__(self, model_id: str = "numind/NuExtract-2.0-2B", use_local_fallback: bool = False):
        self.model_id = model_id
        self.model = None
        self.processor = None
        self.use_fallback = use_local_fallback
        
    def load_model(self):
        if self.use_fallback:
            print("[*] NuExtractRunner operating in lightweight fallback mode.")
            return
            
        print(f"[*] Loading NuExtract 2.0 2B ({self.model_id}) on {device}...")
        try:
            import transformers
            from transformers import AutoProcessor
            
            # Resolve exact Qwen2VL or ConditionalGeneration class for NuExtract 2.0 (Qwen2-VL base)
            AutoModelClass = None
            for class_name in ["Qwen2VLForConditionalGeneration", "AutoModelForConditionalGeneration", "AutoModelForVision2Seq", "AutoModel"]:
                if hasattr(transformers, class_name):
                    AutoModelClass = getattr(transformers, class_name)
                    break
                    
            if AutoModelClass is None:
                raise ImportError("Could not resolve a suitable AutoModel class for Qwen2-VL architecture.")
                
            self.processor = AutoProcessor.from_pretrained(
                self.model_id,
                trust_remote_code=True,
                padding_side='left'
            )
            
            kwargs = {
                "trust_remote_code": True,
                "torch_dtype": torch.bfloat16 if device == "cuda" else torch.float32,
                "device_map": "auto" if device == "cuda" else None
            }
            
            try:
                import flash_attn
                kwargs["attn_implementation"] = "flash_attention_2"
            except ImportError:
                pass
                
            self.model = AutoModelClass.from_pretrained(self.model_id, **kwargs)
            print(f"[✓] Successfully loaded {self.model_id} with {AutoModelClass.__name__}.")
        except Exception as e:
            print(f"[!] Failed to load model from HuggingFace ({e}). Falling back to heuristic extraction.")
            self.use_fallback = True

    def extract(self, text: str, schema: Dict[str, Any], local_context: Optional[str] = None) -> str:
        full_text = f"{local_context}\n\n{text}" if local_context else text
        schema_json = json.dumps(schema, indent=2)
        
        if self.use_fallback or self.model is None:
            return self._heuristic_mock_extract(full_text, schema)
            
        prompt = f"<|input|>\n### Template:\n{schema_json}\n### Text:\n{full_text}\n<|output|>\n"
        inputs = self.processor(text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512, do_sample=False)
        
        input_len = inputs["input_ids"].shape[1]
        output_text = self.processor.decode(outputs[0][input_len:], skip_special_tokens=True)
        return output_text

    def _heuristic_mock_extract(self, text: str, schema: Dict[str, Any]) -> str:
        """Helper rule-based mock matching template fields for instant offline dry runs."""
        res = json.loads(json.dumps(schema))
        lower = text.lower()
        
        dur_match = re.search(r'(\d+|twelve|twenty-four|eighteen|one|two|three)\s*(?:\([^)]+\))?\s*(years?|months?|days?)', lower)
        duration_str = dur_match.group(0) if dur_match else None
        
        if "duration" in res:
            res["duration"] = duration_str
        if "notice_period" in res:
            res["notice_period"] = duration_str
            
        if "jurisdiction" in res:
            if "delaware" in lower:
                res["jurisdiction"] = "State of Delaware"
                res["governing_law"] = "Laws of the State of Delaware"
            elif "new york" in lower:
                res["jurisdiction"] = "State of New York"
                res["governing_law"] = "Laws of New York"
                
        if "restricted_action" in res:
            if "solicit" in lower:
                res["restricted_action"] = "Non-solicitation"
            if "customer" in lower or "client" in lower:
                res["restricted_entities"] = ["customers", "clients"]
            if "employee" in lower or "personnel" in lower:
                res["restricted_entities"] = ["employees", "contractors"]
                
        if "restricted_activity" in res:
            res["restricted_activity"] = "Engaging in competing business activities"
            
        if "protected_information" in res:
            res["protected_information"] = ["trade secrets", "customer lists", "proprietary technical data"]
            res["obligations"] = ["maintain confidentiality", "do not disclose"]
            
        return json.dumps(res)

# Set use_local_fallback=False to run live model loading for numind/NuExtract-2.0-2B
extractor_runner = NuExtractRunner(model_id="numind/NuExtract-2.0-2B", use_local_fallback=True)
extractor_runner.load_model()

In [ ]:
class ExtractionValidator:
    """
    Validation layer for NuExtract JSON output.
    Checks JSON syntax, repairs markdown code blocks, validates schema key matching and types.
    """
    @staticmethod
    def validate_and_repair(raw_output: str, expected_schema: Dict[str, Any]) -> Tuple[bool, Dict[str, Any], List[str]]:
        errors = []
        repaired_json = {}
        
        # 1. Clean markdown formatting
        cleaned = raw_output.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        elif cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
        
        # 2. Safe JSON Parsing
        try:
            parsed = json.loads(cleaned)
        except Exception as e:
            errors.append(f"JSON Parse Failure: {str(e)}")
            return False, expected_schema, errors
            
        if not isinstance(parsed, dict):
            errors.append(f"Root JSON element is {type(parsed).__name__}, expected object/dict.")
            return False, expected_schema, errors
            
        # 3. Schema Keys & Type Compatibility Enforcement
        validated_dict = {}
        for key, expected_default in expected_schema.items():
            if key not in parsed:
                errors.append(f"Missing expected schema key: '{key}'. Set to default ({expected_default}).")
                validated_dict[key] = expected_default
            else:
                val = parsed[key]
                # Type validation for lists
                if isinstance(expected_default, list):
                    if isinstance(val, list):
                        validated_dict[key] = val
                    elif val is None:
                        validated_dict[key] = []
                    elif isinstance(val, str):
                        validated_dict[key] = [val]
                    else:
                        errors.append(f"Key '{key}' expected list, got {type(val).__name__}.")
                        validated_dict[key] = []
                else:
                    validated_dict[key] = val
                    
        is_valid = len(errors) == 0
        return is_valid, validated_dict, errors

print("[✓] Validation layer defined.")

In [ ]:
class Normalizer:
    """
    Normalization layer for extracted legal attributes (e.g. durations, dates).
    """
    WORD_TO_NUM = {
        "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
        "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
        "twelve": 12, "eighteen": 18, "twenty-four": 24, "thirty": 30
    }

    @classmethod
    def normalize_duration(cls, raw_val: Any) -> Any:
        if not raw_val or not isinstance(raw_val, str):
            return raw_val
            
        text = raw_val.lower().strip()
        
        # Extract number / word
        num = None
        match_digit = re.search(r'\b(\d+)\b', text)
        if match_digit:
            num = int(match_digit.group(1))
        else:
            for word, val in cls.WORD_TO_NUM.items():
                if word in text:
                    num = val
                    break
                    
        # Extract unit
        unit = None
        if "year" in text:
            unit = "year"
        elif "month" in text:
            unit = "month"
        elif "day" in text:
            unit = "day"
            
        if num is not None and unit is not None:
            return {
                "raw": raw_val,
                "normalized": {
                    "value": num,
                    "unit": unit
                }
            }
            
        return raw_val

    @classmethod
    def normalize_extraction(cls, category: str, extraction: Dict[str, Any]) -> Dict[str, Any]:
        normalized = dict(extraction)
        # Apply duration normalization where applicable
        for field in ["duration", "notice_period"]:
            if field in normalized and normalized[field] is not None:
                normalized[field] = cls.normalize_duration(normalized[field])
        return normalized

print("[✓] Normalization layer defined.")

In [ ]:
def process_clause_extraction(
    clause_id: str,
    clause_text: str,
    predicted_category: str,
    runner: NuExtractRunner,
    local_context: Optional[str] = None
) -> Dict[str, Any]:
    """
    Full modular pipeline: category -> schema -> NuExtract -> validation -> normalization -> final JSON payload.
    """
    if predicted_category not in EXTRACTION_SCHEMAS:
        raise ValueError(f"Unknown predicted category: '{predicted_category}'")
        
    family = CATEGORY_TO_FAMILY[predicted_category]
    schema = EXTRACTION_SCHEMAS[predicted_category]
    
    # 1. NuExtract Inference
    raw_response = runner.extract(clause_text, schema, local_context=local_context)
    
    # 2. Validation Layer
    is_valid, validated_extraction, validation_errors = ExtractionValidator.validate_and_repair(raw_response, schema)
    
    # 3. Normalization Layer
    normalized_extraction = Normalizer.normalize_extraction(predicted_category, validated_extraction)
    
    # 4. Construct Final Payload
    result_payload = {
        "clause_id": clause_id,
        "category": predicted_category,
        "family": family,
        "text": clause_text,
        "extraction": normalized_extraction,
        "model": {
            "name": f"NuExtract ({runner.model_id})" if not runner.use_fallback else "NuExtract (Fallback/Demo)",
            "version": "2.0-2B",
            "confidence": None
        },
        "validation": {
            "status": "SUCCESS" if is_valid else "REPAIRED_WITH_WARNINGS",
            "errors": validation_errors
        },
        "relationships": []  # Placeholder for cross-clause graph/relationship integration
    }
    
    return result_payload

In [ ]:
# Sample Representative Test Clauses across all 10 categories
TEST_CLAUSES = [
    {
        "id": "CLAUSE_001",
        "category": "confidentiality",
        "text": "The Recipient agrees to hold in strict confidence all proprietary technical data and trade secrets disclosed by the Disclosing Party, and shall not disclose such information for a period of five (5) years."
    },
    {
        "id": "CLAUSE_002",
        "category": "exclusivity",
        "text": "During the term of this Agreement, the Consultant shall work exclusively for the Client and shall not perform similar consulting services for any third party without prior written consent."
    },
    {
        "id": "CLAUSE_003",
        "category": "governing_law",
        "text": "This Agreement shall be governed by, construed, and enforced in accordance with the internal laws of the State of Delaware, without regard to its conflict of laws principles."
    },
    {
        "id": "CLAUSE_004",
        "category": "ip_assignment",
        "text": "Employee hereby irrevocably assigns and transfers to Employer all right, title, and interest in and to all inventions, code, and patents created during the period of employment."
    },
    {
        "id": "CLAUSE_005",
        "category": "liability_indemnity",
        "text": "The Contractor agrees to defend, indemnify, and hold harmless the Company against all third-party claims, liabilities, and legal damages arising from negligence."
    },
    {
        "id": "CLAUSE_006",
        "category": "no_solicit_customers",
        "text": "The Employee agrees that for a period of twelve (12) months following termination of employment, they shall not directly or indirectly solicit or approach any customer or client of the Company."
    },
    {
        "id": "CLAUSE_007",
        "category": "no_solicit_employees",
        "text": "For a period of twenty-four (24) months post-separation, the Executive will not induce or entice away any senior engineers or personnel of the Employer."
    },
    {
        "id": "CLAUSE_008",
        "category": "non_compete",
        "text": "During employment and for eighteen (18) months thereafter, the Employee shall not operate or work for any competing business within North America."
    },
    {
        "id": "CLAUSE_009",
        "category": "other_general",
        "text": "All notices required under this Agreement shall be delivered in writing via certified mail or email to the designated corporate headquarters."
    },
    {
        "id": "CLAUSE_010",
        "category": "termination_notice",
        "text": "Either party may terminate this Agreement without cause by giving at least thirty (30) days prior written notice to the other party."
    }
]

# Execute evaluation run across test dataset
results = []
print("=== Running ClauseGuard NuExtract 2.0 2B Pipeline Evaluation ===\n")
for sample in TEST_CLAUSES:
    res = process_clause_extraction(
        clause_id=sample["id"],
        clause_text=sample["text"],
        predicted_category=sample["category"],
        runner=extractor_runner
    )
    results.append(res)
    print(f"[{res['clause_id']}] Category: {res['category']} | Family: {res['family']} | Status: {res['validation']['status']}")
    print(f"  Extracted JSON: {json.dumps(res['extraction'], indent=2)}")
    print("-" * 80)

In [ ]:
# Summary Report of Pipeline Execution
print("\n=== Pipeline Summary Report ===")
total = len(results)
success_count = sum(1 for r in results if r["validation"]["status"] == "SUCCESS")
warnings_count = total - success_count

print(f"Total Processed Clauses: {total}")
print(f"Clean Success Rate:      {success_count}/{total} ({success_count/total*100:.1f}%)")
print(f"Repaired / Warnings:     {warnings_count}/{total}")

if warnings_count > 0:
    print("\nValidation Warnings Summary:")
    for r in results:
        if r["validation"]["errors"]:
            print(f" - {r['clause_id']} ({r['category']}): {r['validation']['errors']}")